In [33]:
print ("hello pre processing")

hello pre processing


In [34]:
import numpy as np
import pandas as pd
import sklearn 
import matplotlib.pyplot as plt

In [35]:
df = pd.read_csv('../data/ai4i2020.csv')
df

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,M24855,M,298.8,308.4,1604,29.5,14,0,0,0,0,0,0
9996,9997,H39410,H,298.9,308.4,1632,31.8,17,0,0,0,0,0,0
9997,9998,M24857,M,299.0,308.6,1645,33.4,22,0,0,0,0,0,0
9998,9999,H39412,H,299.0,308.7,1408,48.5,25,0,0,0,0,0,0


# حذف ستون‌های غیرضروری و خطرناک

In [36]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
me

In [37]:
cols_to_drop = ['UDI' , 'Product ID' , 'TWF' , 'HDF' , 'PWF' , 'OSF' , 'RNF']

df_clean = df.drop(columns=cols_to_drop)
print(df_clean.shape)
print(df_clean.columns.tolist())

(10000, 7)
['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure']


# تبدیل ستون دسته‌ای Type به عدد (One-Hot Encoding)

In [38]:
df_encoded = pd.get_dummies(df_clean, columns=['Type'], drop_first=True)

print(df_encoded.head())
print(df_encoded.dtypes)

   Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
0                298.1                    308.6                    1551   
1                298.2                    308.7                    1408   
2                298.1                    308.5                    1498   
3                298.2                    308.6                    1433   
4                298.2                    308.7                    1408   

   Torque [Nm]  Tool wear [min]  Machine failure  Type_L  Type_M  
0         42.8                0                0   False    True  
1         46.3                3                0    True   False  
2         49.4                5                0    True   False  
3         39.5                7                0    True   False  
4         40.0                9                0    True   False  
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool we

# جدا کردن X (فیچرها/ورودی‌ها) از y (هدف/خروجی که می‌خوایم پیش‌بینی کنیم)


In [39]:


df_encoded.columns = (
    df_encoded.columns
    .str.replace('[', '_', regex=False)
    .str.replace(']', '_', regex=False)
    .str.replace('<', '_', regex=False)
)

print(df_encoded.head)

<bound method NDFrame.head of       Air temperature _K_  Process temperature _K_  Rotational speed _rpm_  \
0                   298.1                    308.6                    1551   
1                   298.2                    308.7                    1408   
2                   298.1                    308.5                    1498   
3                   298.2                    308.6                    1433   
4                   298.2                    308.7                    1408   
...                   ...                      ...                     ...   
9995                298.8                    308.4                    1604   
9996                298.9                    308.4                    1632   
9997                299.0                    308.6                    1645   
9998                299.0                    308.7                    1408   
9999                299.0                    308.7                    1500   

      Torque _Nm_  Tool wear _min

In [40]:
X = df_encoded.drop(columns=["Machine failure"])
y = df_encoded["Machine failure"]

# تقسیم داده به Train/Test

In [41]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X , y , test_size = 0.2, random_state = 42 , stratify = y)

print (X_train.shape, X_test.shape)
print (y_train.value_counts(normalize = True)*100)
print (y_test.value_counts(normalize = True)*100)

(8000, 7) (2000, 7)
Machine failure
0    96.6125
1     3.3875
Name: proportion, dtype: float64
Machine failure
0    96.6
1     3.4
Name: proportion, dtype: float64


# نرمال‌سازی (StandardScaler)

In [44]:
from sklearn.preprocessing import StandardScaler

numeric_cols= ['Air temperature _K_', 'Process temperature _K_','Rotational speed _rpm_', 'Torque _Nm_', 'Tool wear _min_']

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(X_train_scaled[numeric_cols].describe())

       Air temperature _K_  Process temperature _K_  Rotational speed _rpm_  \
count         8.000000e+03             8.000000e+03            8.000000e+03   
mean          1.758593e-15             1.053468e-14            3.010925e-16   
std           1.000063e+00             1.000063e+00            1.000063e+00   
min          -2.356591e+00            -2.910801e+00           -2.052017e+00   
25%          -8.541261e-01            -8.152711e-01           -6.484822e-01   
50%           4.735268e-02             6.349964e-02           -2.008982e-01   
75%           7.485028e-01             7.394771e-01            4.069319e-01   
max           2.250967e+00             2.564616e+00            7.441184e+00   

        Torque _Nm_  Tool wear _min_  
count  8.000000e+03     8.000000e+03  
mean  -1.474376e-16    -3.064216e-17  
std    1.000063e+00     1.000063e+00  
min   -3.613500e+00    -1.692947e+00  
25%   -6.790515e-01    -8.597185e-01  
50%    9.645501e-03    -1.076908e-02  
75%    6.783803

# SMOTE و متوازن سازی داده ها در قسمت میزان خرابی و سالم بودن

In [45]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(X_train_scaled.shape)
print (X_train_resampled.shape)
print (y_train_resampled.value_counts())
print (y_train_resampled.value_counts(normalize = True)*100)

(8000, 7)
(15458, 7)
Machine failure
0    7729
1    7729
Name: count, dtype: int64
Machine failure
0    50.0
1    50.0
Name: proportion, dtype: float64


# مدل پایه (Baseline)

### پیاده‌سازی Logistic Regression ساده (بدون تنظیم پارامتر خاص) به‌عنوان نقطه‌ی مرجع

In [46]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_resampled , y_train_resampled)

y_pred_log = log_reg.predict(X_test_scaled)
print (y_pred_log[:20])
print (y_test.values[:20])


[1 1 1 0 0 0 1 0 0 0 0 0 0 0 0 1 0 1 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]


In [47]:
from sklearn.metrics import accuracy_score

acc_test_log = accuracy_score(y_test , y_pred_log)

acc_test_log

0.839

# Random forest and xgboost

In [48]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_resampled, y_train_resampled)
y_pred_rf = rf.predict(X_test_scaled)

xgb = XGBClassifier(random_state=42)
xgb.fit(X_train_resampled, y_train_resampled)
y_pred_xgb = xgb.predict(X_test_scaled)

print(y_pred_rf[:20])
print(y_pred_xgb[:20])
print(y_test.values[:20])

[0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]


In [50]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_rec = recall_score(y_test, y_pred_rf)
rf_pre = precision_score(y_test, y_pred_rf)
print (rf_acc , rf_rec , rf_pre)

xgb_acc = accuracy_score(y_test, y_pred_xgb)
xgb_rec = recall_score(y_test, y_pred_xgb)
xgb_pre = precision_score(y_test, y_pred_xgb)
print(xgb_acc , xgb_rec , xgb_pre)

0.9665 0.7647058823529411 0.5048543689320388
0.9785 0.8088235294117647 0.6470588235294118
